<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 36px 32px; border-radius: 16px; margin-bottom: 8px;">
  <h1 style="color: #fff; font-family: 'Georgia', serif; font-size: 2.2em; margin: 0 0 8px 0; letter-spacing: -1px;">🔗 Week 5: Backend Integration Pipeline</h1>
  <p style="color: #7dd3fc; font-size: 1.1em; margin: 0; font-family: monospace;">Audio Capture → Whisper STT → Speaker Diarization → LLM Summarization</p>
  <hr style="border-color: #1e4a6a; margin: 20px 0 16px 0;">
  <table style="color: #e0f2fe; font-size: 0.95em; font-family: monospace; border-collapse: collapse;">
    <tr><td style="padding: 4px 16px 4px 0;">📦 Step 1</td><td>Install all dependencies</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🔑 Step 2</td><td>Enter Groq API key</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🎵 Step 3</td><td>Upload audio file (.wav / .mp3)</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🧵 Step 4</td><td>Preprocess audio (mono, 16kHz)</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">📝 Step 5</td><td>Whisper Speech-to-Text (threaded)</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">👥 Step 6</td><td>Speaker Diarization (ECAPA-TDNN + KMeans)</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🤖 Step 7</td><td>LLM Summarization via Groq (runs after STT+Diarization)</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">📊 Step 8</td><td>ROUGE Evaluation</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">💾 Step 9</td><td>Save & Download outputs</td></tr>
  </table>
</div>

## 📦 Step 1 — Install Dependencies

In [ ]:
print("Installing packages... (takes ~2 min on first run)")

# Core audio & ML
!pip install -q \
    torch==2.3.1+cu121 \
    torchaudio==2.3.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

!pip install -q "huggingface_hub==0.23.4"
!pip install -q "speechbrain==1.0.1"
!pip install -q "pyannote.metrics==3.2.1"
!pip install -q "pyannote.core==5.0.0"
!pip install -q openai-whisper
!pip install -q soundfile scikit-learn

# Summarization & evaluation
!pip install -q groq rouge-score pydub ipywidgets
!apt-get install -y ffmpeg -qq

print("\n✅ All packages ready! Go to Runtime > Restart session, then re-run from Step 2.")

## 🔑 Step 2 — Enter Groq API Key
> Get a free key at **https://console.groq.com** → API Keys

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os, time, threading, queue, json
from datetime import datetime
import numpy as np

display(HTML("<p style='font-size:1em; color:#555;'>🔑 Paste your Groq API key below:</p>"))

api_key_widget = widgets.Password(
    placeholder='gsk_...',
    layout=widgets.Layout(width='440px', height='38px')
)
display(api_key_widget)
display(HTML("<p style='color:#888; font-size:0.85em; margin-top:6px;'>🔒 Hidden input — key is not displayed</p>"))

## 🎵 Step 3 — Upload Your Audio File

In [ ]:
from google.colab import files

display(HTML("""
<div style='background:#f0f7ff; border:2px dashed #4f8ef7; border-radius:12px;
     padding:24px 28px; margin:8px 0; text-align:center;'>
  <div style='font-size:2.4em; margin-bottom:6px;'>🎙️</div>
  <div style='font-size:1.1em; font-weight:bold; color:#1a56db;'>Upload your meeting audio</div>
  <div style='color:#555; font-size:0.9em; margin-top:4px;'>Supported: .wav &nbsp;|&nbsp; .mp3 &nbsp;|&nbsp; .m4a &nbsp;|&nbsp; .flac</div>
</div>
"""))

uploaded = files.upload()

if uploaded:
    AUDIO_FILE = list(uploaded.keys())[0]
    size_kb = os.path.getsize(AUDIO_FILE) / 1024
    display(HTML(f"""
    <div style='background:#f0fff4; border:1px solid #34d399; border-radius:8px; padding:14px 18px; margin-top:10px;'>
      <b style='color:#065f46;'>✅ File uploaded!</b><br>
      <span style='font-family:monospace; color:#047857;'>📁 {AUDIO_FILE}</span>
      <span style='color:#6b7280; font-size:0.88em;'> ({size_kb:.1f} KB)</span>
    </div>
    """))
else:
    AUDIO_FILE = None
    print("⚠️ No file uploaded yet.")

## 🧵 Step 4 — Preprocess Audio (Mono + 16kHz)
> Converts the uploaded file to the standard format needed by both Whisper and the speaker encoder.

In [ ]:
import torch
import torchaudio

waveform, orig_sr = torchaudio.load(AUDIO_FILE)
print(f"Original  : shape={waveform.shape}  sr={orig_sr}")

# Mono
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# Resample to 16kHz
TARGET_SR = 16000
if orig_sr != TARGET_SR:
    waveform = torchaudio.transforms.Resample(orig_sr, TARGET_SR)(waveform)

MEETING_ID     = os.path.splitext(os.path.basename(AUDIO_FILE))[0]
PROCESSED_PATH = f"{MEETING_ID}_16k.wav"
torchaudio.save(PROCESSED_PATH, waveform, TARGET_SR)

duration = waveform.shape[1] / TARGET_SR
print(f"Processed : shape={waveform.shape}  sr={TARGET_SR}")
print(f"Duration  : {duration:.1f}s  ({int(duration//60)}m {int(duration%60)}s)")
print(f"Saved     : {PROCESSED_PATH}")
print("✅ Audio ready — proceed to Step 5")

## 📝 Step 5 — Whisper STT (Runs in a Background Thread)
> **Week 5 key concept:** STT runs in a dedicated thread so it does not block the UI.  
> Results are stored in a shared `Queue` and consumed by Step 7 only after both STT and Diarization are complete.

In [ ]:
import whisper

# ── Shared state (queue + event flags) ───────────────────────────
stt_result_queue   = queue.Queue()   # stores final transcript dict
stt_done_event     = threading.Event()  # signals STT completion
diarize_done_event = threading.Event()  # signals diarization completion

TRANSCRIPT_TEXT  = ""   # plain text transcript
STT_WORDS        = []   # word-level timestamps
STT_SEGMENTS     = []   # segment-level results

# ── Model selector ───────────────────────────────────────────────
display(HTML("<p style='font-size:1em; color:#444;'>🔧 Choose Whisper model size:</p>"))
whisper_model_widget = widgets.RadioButtons(
    options=[
        ('tiny  — Fastest (lower accuracy)', 'tiny'),
        ('base  — Recommended ✅', 'base'),
        ('small — Better accuracy (slower)', 'small'),
    ],
    value='base',
    layout=widgets.Layout(width='380px')
)
display(whisper_model_widget)

stt_btn    = widgets.Button(description='🎙️ Start STT Thread',
                             button_style='primary',
                             layout=widgets.Layout(width='220px', height='44px', margin='10px 0 0 0'))
stt_output = widgets.Output()

def stt_worker():
    """Runs in background thread — stores result in queue."""
    global TRANSCRIPT_TEXT, STT_WORDS, STT_SEGMENTS
    try:
        model_size = whisper_model_widget.value
        model      = whisper.load_model(model_size)
        result     = model.transcribe(PROCESSED_PATH, word_timestamps=True)

        TRANSCRIPT_TEXT = result["text"].strip()
        STT_SEGMENTS    = result["segments"]

        for seg in result["segments"]:
            for w in seg.get("words", []):
                STT_WORDS.append({
                    "word":  w["word"].strip(),
                    "start": round(w["start"], 3),
                    "end":   round(w["end"],   3),
                })

        stt_result_queue.put({"text": TRANSCRIPT_TEXT, "words": STT_WORDS})
        stt_done_event.set()    # ← signal: STT done
    except Exception as e:
        stt_result_queue.put({"error": str(e)})
        stt_done_event.set()

def start_stt(b):
    with stt_output:
        clear_output(wait=True)
        display(HTML("<p>⏳ Launching STT thread — this runs in the background...</p>"))
        t = threading.Thread(target=stt_worker, daemon=True)
        t.start()

        # Poll until done (non-blocking in colab via widget output)
        t.join()  # wait for thread to finish before printing result
        result = stt_result_queue.get()

        if "error" in result:
            display(HTML(f"<p style='color:red;'>❌ STT Error: {result['error']}</p>"))
            return

        wc = len(TRANSCRIPT_TEXT.split())
        display(HTML(f"""
        <div style='background:#f0fff4; border:1px solid #6ee7b7; border-radius:10px; padding:14px 18px;'>
          <b style='color:#065f46;'>✅ STT complete — {wc} words transcribed</b><br>
          <span style='color:#555; font-size:0.9em;'>Result stored in queue ✔ | stt_done_event set ✔</span>
        </div>
        """))
        print("\n📄 Transcript preview:")
        print(TRANSCRIPT_TEXT[:400], "..." if len(TRANSCRIPT_TEXT) > 400 else "")

stt_btn.on_click(start_stt)
display(stt_btn)
display(stt_output)

## 👥 Step 6 — Speaker Diarization (Runs in a Background Thread)
> ECAPA-TDNN embeddings + KMeans clustering, same approach as Week 3 — now wrapped in a thread.

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from collections import defaultdict

HYP_SEGMENTS = []   # shared: filled by diarization thread

display(HTML("<p style='font-size:1em; color:#444;'>👥 How many speakers are in the recording?</p>"))
num_speakers_widget = widgets.BoundedIntText(
    value=2, min=1, max=10,
    description='Speakers:',
    layout=widgets.Layout(width='200px')
)
display(num_speakers_widget)

diarize_btn    = widgets.Button(description='👥 Start Diarization Thread',
                                 button_style='warning',
                                 layout=widgets.Layout(width='260px', height='44px', margin='10px 0 0 0'))
diarize_output = widgets.Output()

def smooth_labels(labels, window=5):
    smoothed = labels.copy()
    for i in range(len(labels)):
        s = max(0, i - window // 2)
        e = min(len(labels), i + window // 2 + 1)
        smoothed[i] = np.bincount(labels[s:e]).argmax()
    return smoothed

def diarization_worker():
    global HYP_SEGMENTS
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        encoder = EncoderClassifier.from_hparams(
            source="speechbrain/spkrec-ecapa-voxceleb",
            savedir="pretrained_models/spkrec-ecapa-voxceleb",
            run_opts={"device": device},
        )
        encoder.eval()

        WINDOW_S    = 500
        HOP_S       = 250
        win_samples = int(WINDOW_S * TARGET_SR)
        hop_samples = int(HOP_S * TARGET_SR)
        waveform_np = waveform.squeeze().numpy()

        windows, window_times = [], []
        i = 0
        while i + win_samples <= len(waveform_np):
            windows.append(waveform_np[i:i + win_samples])
            window_times.append({"start": round(i / TARGET_SR, 3),
                                  "end":   round((i + win_samples) / TARGET_SR, 3)})
            i += hop_samples

        embeddings = []
        with torch.no_grad():
            for chunk in windows:
                t = torch.tensor(chunk).unsqueeze(0).float().to(device)
                embeddings.append(encoder.encode_batch(t).squeeze().cpu().numpy())
        embeddings = np.array(embeddings)

        NUM_SPEAKERS = num_speakers_widget.value
        emb_norm = normalize(embeddings)
        labels   = KMeans(n_clusters=NUM_SPEAKERS, random_state=42, n_init=10).fit_predict(emb_norm)
        labels   = smooth_labels(labels)

        # Build segments
        segs = []
        seg_start, seg_label = window_times[0]["start"], labels[0]
        for i in range(1, len(window_times)):
            if labels[i] != seg_label:
                segs.append({"speaker": f"speaker{seg_label+1}",
                              "start": seg_start, "end": window_times[i-1]["end"]})
                seg_start, seg_label = window_times[i]["start"], labels[i]
        segs.append({"speaker": f"speaker{seg_label+1}",
                     "start": seg_start, "end": window_times[-1]["end"]})

        # Merge short segments
        merged = []
        for seg in segs:
            if seg["end"] - seg["start"] < 0.5 and merged:
                merged[-1]["end"] = seg["end"]
            else:
                merged.append(dict(seg))

        HYP_SEGMENTS = merged
        diarize_done_event.set()   # ← signal: diarization done
    except Exception as e:
        print(f"❌ Diarization error: {e}")
        diarize_done_event.set()

def start_diarization(b):
    with diarize_output:
        clear_output(wait=True)
        display(HTML("<p>⏳ Launching diarization thread...</p>"))
        t = threading.Thread(target=diarization_worker, daemon=True)
        t.start()
        t.join()

        if not HYP_SEGMENTS:
            display(HTML("<p style='color:red;'>❌ Diarization failed or produced no segments.</p>"))
            return

        spk_dur = defaultdict(float)
        for seg in HYP_SEGMENTS:
            spk_dur[seg["speaker"]] += seg["end"] - seg["start"]
        total = sum(spk_dur.values()) or 1

        display(HTML(f"""
        <div style='background:#fffbeb; border:1px solid #fbbf24; border-radius:10px; padding:14px 18px;'>
          <b style='color:#92400e;'>✅ Diarization complete — {len(HYP_SEGMENTS)} turns detected</b><br>
          <span style='color:#555; font-size:0.9em;'>diarize_done_event set ✔ | Ready for merge in Step 7</span>
        </div>
        """))
        print(f"\n{'SPEAKER':<14} {'DURATION':>10}   {'SHARE':>7}")
        print("-" * 40)
        for spk, dur in sorted(spk_dur.items()):
            pct = 100 * dur / total
            print(f"{spk:<14} {int(dur//60):02d}m{dur%60:05.2f}s   {pct:6.1f}%")

diarize_btn.on_click(start_diarization)
display(diarize_btn)
display(diarize_output)

## 🤖 Step 7 — Merge + Summarize (Runs ONLY After STT & Diarization Are Done)
> **Week 5 key concept:** Summarization is gated behind both `stt_done_event` and `diarize_done_event`.  
> This guarantees no overlapping processes and no crashes.

In [ ]:
MERGED_TRANSCRIPT = []   # speaker-labelled transcript segments
LAST_SUMMARY      = ""

PROMPT_TEMPLATES = {
    "📋 Full Summary (Overview + Key Points + Decisions + Action Items)": """You are a professional meeting summarizer. Analyze the transcript and return a structured summary in EXACTLY this format:

**MEETING SUMMARY**

**Overview:**
[2-3 sentence overview]

**Key Points:**
- [point]

**Decisions Made:**
- [decision]

**Action Items:**
| Owner | Task | Deadline |
|-------|------|----------|
| [name] | [task] | [deadline or TBD] |

**Next Steps:**
- [step]

TRANSCRIPT:
{transcript}""",

    "🔑 Key Points Only": """Extract the KEY POINTS from this meeting transcript as a clear bullet list.

**Key Points:**
- [point]

TRANSCRIPT:
{transcript}""",

    "✅ Action Items Only": """Extract all ACTION ITEMS from this meeting transcript.

| Owner | Task | Deadline |
|-------|------|----------|
| [name] | [task] | [deadline] |

TRANSCRIPT:
{transcript}""",
}

display(HTML("<p style='font-size:1em; color:#444;'><b>📋 Choose summary type:</b></p>"))
summary_type_widget = widgets.RadioButtons(
    options=list(PROMPT_TEMPLATES.keys()),
    value=list(PROMPT_TEMPLATES.keys())[0],
    layout=widgets.Layout(width='600px')
)
display(summary_type_widget)

summarize_btn    = widgets.Button(
    description='⚡ Merge + Summarize',
    button_style='success',
    layout=widgets.Layout(width='240px', height='44px', margin='14px 0 0 0')
)
summarize_output = widgets.Output()

def merge_stt_diarization(hyp_segments, stt_words):
    """Align word-level STT output with speaker diarization segments."""
    transcript = []
    for seg in hyp_segments:
        words_in_seg = [
            w["word"] for w in stt_words
            if w["start"] >= seg["start"] and w["end"] <= seg["end"]
        ]
        transcript.append({
            "speaker": seg["speaker"],
            "start":   seg["start"],
            "end":     seg["end"],
            "text":    " ".join(words_in_seg).strip() or "[inaudible]",
        })
    return transcript

def summarization_worker(transcript_text, api_key, summary_type):
    """Called in a thread — only after both events are set."""
    global LAST_SUMMARY
    from groq import Groq
    prompt  = PROMPT_TEMPLATES[summary_type].format(transcript=transcript_text)
    client  = Groq(api_key=api_key)
    resp    = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3,
        max_tokens=1500
    )
    LAST_SUMMARY = resp.choices[0].message.content

def run_pipeline(b):
    global MERGED_TRANSCRIPT, LAST_SUMMARY
    with summarize_output:
        clear_output(wait=True)

        # ── Gate: wait for both upstream tasks ──────────────────
        display(HTML("<p>🔍 Checking pipeline status...</p>"))

        stt_ready     = stt_done_event.is_set()
        diarize_ready = diarize_done_event.is_set()

        display(HTML(f"""
        <div style='font-family:monospace; font-size:0.9em; padding:10px; background:#1e1e2e;
             color:#cdd6f4; border-radius:8px; margin:8px 0;'>
          STT done event   : {'✅ SET' if stt_ready else '❌ NOT SET — run Step 5 first'}<br>
          Diarize done event: {'✅ SET' if diarize_ready else '❌ NOT SET — run Step 6 first'}
        </div>
        """))

        if not stt_ready or not diarize_ready:
            display(HTML("<p style='color:red;'>❌ Cannot summarize yet. Complete Steps 5 and 6 first.</p>"))
            return

        # ── Merge STT + Diarization ──────────────────────────────
        display(HTML("<p>🔗 Merging speaker labels with transcript words...</p>"))
        MERGED_TRANSCRIPT = merge_stt_diarization(HYP_SEGMENTS, STT_WORDS)

        # Build speaker-tagged text for the LLM
        labelled_text = ""
        for seg in MERGED_TRANSCRIPT:
            if seg["text"] != "[inaudible]":
                labelled_text += f"{seg['speaker']}: {seg['text']}\n"

        display(HTML("<p>📋 Speaker-tagged transcript preview:</p>"))
        print(labelled_text[:600], "..." if len(labelled_text) > 600 else "")

        # ── Run summarization in a thread ────────────────────────
        api_key = api_key_widget.value.strip()
        if not api_key:
            display(HTML("<p style='color:red;'>❌ Please enter your Groq API key in Step 2.</p>"))
            return

        display(HTML("<p>🚀 Running summarization (Groq LLaMA 3.1)...</p>"))
        try:
            t = threading.Thread(
                target=summarization_worker,
                args=(labelled_text, api_key, summary_type_widget.value),
                daemon=True
            )
            t.start()
            t.join()  # wait safely
        except Exception as e:
            display(HTML(f"<p style='color:red;'>❌ Summarization error: {e}</p>"))
            return

        display(HTML("""
        <div style='background:#f0fff4; border:1px solid #34d399; border-radius:10px;
             padding:12px 18px; margin:8px 0; color:#065f46; font-weight:bold;'>
          ✅ Summary generated successfully!
        </div>
        """))
        print("\n" + "═"*62)
        print(LAST_SUMMARY)
        print("═"*62)

summarize_btn.on_click(run_pipeline)
display(summarize_btn)
display(summarize_output)

## 📊 Step 8 — ROUGE Evaluation
> Paste a human reference summary to score the AI output.  
> *(Carried forward from Week 4)*

In [ ]:
from rouge_score import rouge_scorer as rs

display(HTML("<p style='font-size:1em; color:#444;'>📝 Paste a human-written reference summary (ground truth):</p>"))
reference_widget = widgets.Textarea(
    placeholder='Write or paste a reference summary here...',
    layout=widgets.Layout(width='100%', height='130px')
)
display(reference_widget)

rouge_btn    = widgets.Button(description='📊 Calculate ROUGE Scores',
                               button_style='info',
                               layout=widgets.Layout(width='250px', height='44px', margin='10px 0 0 0'))
rouge_output = widgets.Output()

def run_rouge(b):
    with rouge_output:
        clear_output(wait=True)
        if not LAST_SUMMARY:
            display(HTML("<p style='color:red;'>❌ No summary yet — run Step 7 first.</p>"))
            return
        reference = reference_widget.value.strip()
        if not reference:
            display(HTML("<p style='color:orange;'>⚠️ Please enter a reference summary above.</p>"))
            return

        scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        scores = scorer.score(reference, LAST_SUMMARY)

        print("\n" + "═"*54)
        print("   📊 ROUGE EVALUATION RESULTS")
        print("═"*54)
        results = [
            ("ROUGE-1", scores['rouge1'],  0.60, "Word overlap"),
            ("ROUGE-2", scores['rouge2'],  0.35, "Phrase overlap"),
            ("ROUGE-L", scores['rougeL'],  0.50, "Sequence match"),
        ]
        f1_scores = []
        for name, score, threshold, desc in results:
            p  = round(score.precision, 3)
            r  = round(score.recall,    3)
            f1 = round(score.fmeasure,  3)
            f1_scores.append(f1)
            status = "✅ PASS" if f1 >= threshold else "⚠️  LOW"
            filled = int(f1 * 28)
            bar    = "█" * filled + "░" * (28 - filled)
            print(f"\n  {name}  [{bar}]  F1 = {f1}")
            print(f"  {desc}  |  P={p}  R={r}  |  Threshold ≥{threshold}  {status}")

        avg = round(sum(f1_scores) / 3, 3)
        print("\n" + "─"*54)
        print(f"  Average F1:  {avg}")
        if avg >= 0.50:   print("  🏆 Overall: GOOD — strong summary quality!")
        elif avg >= 0.35: print("  🔶 Overall: ACCEPTABLE — try refining the prompt")
        else:             print("  🔴 Overall: LOW — try a larger model or better prompt")
        print("═"*54)

rouge_btn.on_click(run_rouge)
display(rouge_btn)
display(rouge_output)

## 💾 Step 9 — Save & Download All Outputs

In [ ]:
from google.colab import files as colab_files

fname_widget = widgets.Text(value='meeting_output', description='Filename:',
                             layout=widgets.Layout(width='300px'))
fmt_widget   = widgets.Dropdown(
    options=[('.txt — Plain text', 'txt'), ('.md — Markdown', 'md'), ('.json — Full JSON', 'json')],
    value='txt', layout=widgets.Layout(width='260px')
)
save_btn    = widgets.Button(description='💾 Download Outputs',
                              button_style='primary',
                              layout=widgets.Layout(width='220px', height='42px', margin='10px 0 0 0'))
save_output = widgets.Output()

def save_and_download(b):
    with save_output:
        clear_output(wait=True)
        if not LAST_SUMMARY:
            display(HTML("<p style='color:red;'>❌ No summary to save — run Step 7 first.</p>"))
            return

        fmt  = fmt_widget.value.split()[0].replace('.', '')
        name = fname_widget.value.strip() or 'meeting_output'
        ts   = datetime.now().strftime("%Y-%m-%d %H:%M")

        files_to_download = []

        # 1. Summary file
        summary_file = f"{name}_summary.{fmt}"
        if fmt == 'json':
            data = {
                "timestamp":          ts,
                "audio_file":         AUDIO_FILE or "unknown",
                "summary_type":       summary_type_widget.value,
                "plain_transcript":   TRANSCRIPT_TEXT,
                "merged_transcript":  MERGED_TRANSCRIPT,
                "summary":            LAST_SUMMARY,
            }
            with open(summary_file, 'w') as f:
                json.dump(data, f, indent=2)
        else:
            header = f"Meeting Summary — {ts}\nAudio: {AUDIO_FILE or 'n/a'}\n{'='*60}\n\n"
            with open(summary_file, 'w') as f:
                f.write(header + LAST_SUMMARY)
        files_to_download.append(summary_file)

        # 2. Speaker-labelled transcript .txt
        transcript_file = f"{name}_transcript.txt"
        with open(transcript_file, 'w') as f:
            f.write(f"Meeting : {MEETING_ID}\n" + "="*65 + "\n\n")
            for seg in MERGED_TRANSCRIPT:
                if seg["text"] == "[inaudible]": continue
                f.write(f"{seg['speaker']}:\n")
                f.write(f"  [{int(seg['start']//60):02d}:{seg['start']%60:05.2f} --> "
                        f"{int(seg['end']//60):02d}:{seg['end']%60:05.2f}]\n")
                f.write(f"  {seg['text']}\n\n")
        files_to_download.append(transcript_file)

        # 3. Raw diarization JSON (same as week 3)
        diarize_file = f"{name}_diarization.json"
        with open(diarize_file, 'w') as f:
            json.dump(HYP_SEGMENTS, f, indent=2)
        files_to_download.append(diarize_file)

        display(HTML(f"<p style='color:green;'>✅ Downloading {len(files_to_download)} files...</p>"))
        for fname in files_to_download:
            colab_files.download(fname)
            print(f"  ⬇  {fname}")
        print("\n✅ All outputs downloaded!")

save_btn.on_click(save_and_download)
display(widgets.HBox([fname_widget, fmt_widget]))
display(save_btn)
display(save_output)

---
<div style='background:#1e1e2e; border-radius:12px; padding:20px 24px; color:#cdd6f4;'>
<h3 style='color:#cba6f7; margin-top:0;'>🏗️ Week 5 — Full Backend Pipeline Architecture</h3>
<pre style='color:#a6e3a1; background:transparent; margin:0;'>
.wav / .mp3
     │
     ▼
  Preprocess (mono → 16kHz)
     │
     ├──── Thread 1: Whisper STT ──► stt_done_event.set()
     │                                        │
     └──── Thread 2: Diarization ──► diarize_done_event.set()
                                              │
                    ┌─────────────────────────┘
                    │  (gated — waits for BOTH events)
                    ▼
          Merge STT + Speaker Labels
                    │
                    ▼
          Thread 3: Groq Summarization
                    │
                    ▼
          ROUGE Evaluation
                    │
                    ▼
     📄 transcript.txt  📊 diarization.json  📋 summary.txt
</pre>
<p style='color:#89b4fa; margin-top:12px; font-size:0.9em;'>
Key concepts used: <code>threading.Thread</code> · <code>queue.Queue</code> · <code>threading.Event</code> · gated execution
</p>
</div>